In [ ]:
# 1/4 - clone main + prototype (fresh, fault-tolerant)
import subprocess, os
print('===== NATIVE GeluErfFwd BITEXACT TEST (PURE TKV, AVX2-replica + scalar tail) =====')
for d,br in [('/content/TVmain','main'),('/content/TVcpu','prototype/cuda-backend')]:
    subprocess.run('rm -rf '+d, shell=True)
    subprocess.run('git clone --depth 1 --branch '+br+' https://github.com/nguyenhungtran18/TokenVector.git '+d, shell=True, capture_output=True, text=True, timeout=600)
    print(br, subprocess.check_output('git -C '+d+' rev-parse --short HEAD', shell=True).decode().strip())

In [ ]:
# 2/4 - build native .so + .tkv C .so. NO C injection. CUNG flags.
import subprocess, os
FLAGS='-O2 -mavx2 -mfma'
root='/content/TVmain/Release/tools/simd_native'
srcs='tkv_simd_native.c tkv_cuda_native.c tkv_linear_dw.c tkv_bwd_ops.c tkv_flash_attn.c tkv_step124.c tkv_prof.c tkv_attn_cuda.c'
p=subprocess.run('cd '+root+' && gcc '+FLAGS+' -shared -fPIC '+srcs+' -lm -lpthread -ldl -o tkv_simd_native.so', shell=True, capture_output=True, text=True, timeout=600)
print('native build rc=',p.returncode)
if p.returncode!=0: print(p.stderr[:2000])
ex='/content/TVcpu/Release/cuda_backend/examples'
cc=ex+'/gelu_erf_fwd_pure.cpu.c'; so=ex+'/gelu_erf_fwd_pure.so'
txt=open(cc).read()
print('C INJECTED =', 'tkv_gelu_f' in txt, '(expect False)')
print('rint/ldexp =', 'rintf(' in txt and 'ldexpf(' in txt, ' d_exp =', 'exp(' in txt)
p2=subprocess.run('gcc '+FLAGS+' -shared -fPIC '+cc+' -o '+so+' -lm -lpthread -ldl', shell=True, capture_output=True, text=True, timeout=600)
print('tkv C build rc=',p2.returncode)
if p2.returncode!=0: print(p2.stderr[:2000])

In [ ]:
# 3/4 - bitwise compare GeluErfFwd
import ctypes, numpy as np
PP=ctypes.POINTER(ctypes.c_float); II=ctypes.c_int; LL=ctypes.c_longlong
args=[PP,II,PP,II,LL,LL]
native=ctypes.CDLL('/content/TVmain/Release/tools/simd_native/tkv_simd_native.so').GeluErfFwd
native.argtypes=args; native.restype=II
tkv=ctypes.CDLL('/content/TVcpu/Release/cuda_backend/examples/gelu_erf_fwd_pure.so').GeluErfFwd
tkv.argtypes=args; tkv.restype=II
rng=np.random.default_rng(41)
for lo,hi in [(0,64),(5,64),(0,1000),(3,777)]:
    x=rng.standard_normal(hi).astype(np.float32)
    a=np.zeros(hi-lo,np.float32); c=np.zeros(hi-lo,np.float32)
    native(x.ctypes.data_as(PP),hi,a.ctypes.data_as(PP),a.size,lo,hi)
    tkv  (x.ctypes.data_as(PP),hi,c.ctypes.data_as(PP),c.size,lo,hi)
    d=np.count_nonzero(a.view(np.uint32)!=c.view(np.uint32))
    print('lo=%d hi=%d bitwise=%s diff=%d/%d maxabs=%.3g'%(lo,hi,d==0,d,a.size,np.abs(a-c).max()))

In [ ]:
# 4/4 - summary
print('If all bitwise=True -> GeluErfFwd pure .tkv (AVX2 replica + scalar tail) replaces auto-glue C.')